In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import DRT_PATH, TP4_PATH, MODEL_PATH, COLUMN_PATH

ABS_PATH = PROJECT_ROOT + DRT_PATH + TP4_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + COLUMN_PATH

model_name = "column.blend"

In [ ]:
# Create needed folders
import os

lst_folders = ["figures", "files", "models"]

for name in lst_folders:
    os.makedirs(os.path.join(ABS_PATH, name), exist_ok=True)

In [ ]:
import torch
import pandas as pd

from modelaquisition.bl2pina import Blend2Pina
from modelaquisition.bl2msh import Blend2Mesh
from modelaquisition.msh2xdmf import Msh2Xdmf

In [ ]:
torch.set_default_dtype(torch.float64)

# Sistema di PDE in considerazione

Consideriamo il problema sul dominio $\Omega$ di frontiera $\Gamma = \partial \Omega$
$$
\begin{equation}
    \begin{cases}
        \Delta u = & \left(
                \begin{matrix}
                    2 u_1\\
                    2
                \end{matrix}
            \right), & \left(x,y,z\right) \in \Omega \\
        u \left( x, y, z \right) = & \left(
                \begin{matrix}
                    e^{x+y}\\
                    x^2 - z
                \end{matrix} \right) & \left( x, y, z \right) \in \Gamma
    \end{cases}
    \tag{1}
\end{equation}
$$

di soluzione analitica $u : \Omega \subseteq \mathbb{R}^3 \longmapsto \mathbb{R}^2$
$$
\begin{equation}
    u \left( x, y, z \right) = \left(
                \begin{matrix}
                    e^{x+y}\\
                    x^2 - z
                \end{matrix} \right) \qquad \left( x, y, z \right) \in \Omega
    \tag{2}
\end{equation}
$$

## Creazione dei dati al contorno

In [ ]:
column = Blend2Pina(LOAD_MODEL + model_name)

num_points_int = 10_000
num_points_b = 10_000

domain = column.intern()
surface = column.boundary()
points_internal = domain.sample(num_points_int)
points_boundary = surface.sample(num_points_b)

Memorizziamo i punti interni

In [ ]:
df_int = pd.DataFrame(
    points_internal.tensor.detach().numpy()
)
df_int.to_csv("./files/input_int.csv", sep = ";")

In [ ]:
u1 = torch.exp(points_boundary.extract('x') + points_boundary.extract('y'))
u2 = (points_boundary.extract('x')**2 - points_boundary.extract('z'))

In [ ]:
print(f"Valore massimo u1: {torch.max(u1).tensor.item()}")
print(f"Valore massimo u2: {torch.max(u2).tensor.item()}")

In [ ]:
total_info = torch.concat(
    [points_boundary.tensor, u1, u2],
    1
)

df = pd.DataFrame(
    data=total_info.numpy(),
    columns=["x", "y", "z", "u1", "u2"]
)

df.to_csv("./files/data.csv", sep=";")

## Creazione mesh e xdmf

In [ ]:
column_msh = Blend2Mesh(LOAD_MODEL + model_name, "column")

In [ ]:
column_msh.create_single_meshes(len_msh=.01)

In [ ]:
column_xdmf = Msh2Xdmf("column.msh", "column")
column_xdmf.to_xdmf(num_refine=3)
column_xdmf.to_xdmf()